# Pre-tutorial Setup

In [ ]:
import pandas as pd
import duckdb

# Load iris dataset from public source
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv")

# Save as CSV using pandas
df.to_csv("iris.csv", index=False)

# Save as DuckDB database
con = duckdb.connect("iris.db")
con.execute("CREATE OR REPLACE TABLE iris AS SELECT * FROM df")
con.close()

print(f"Setup complete: iris.csv and iris.db created ({len(df)} rows, {df.shape[1]} columns)")

# Pandas for SQL Users

Welcome to the **Pandas for SQL Users** tutorial. The goal of this tutorial is to give a quick introduction to the **Pandas** Python package for people familiar with SQL and the basics of Python.

*(Note: This tutorial uses Jupyter notebooks which are a way to split Python programs into smaller "cells" and run them step by step, adding text cells for comments in between)*

To illustrate the concepts discussed here, this environment contains two files:
* `iris.csv`
* `iris.db`

The `iris.csv` file is a regular CSV which you can open with, e.g., Microsoft Excel. It contains a famous dataset of measurements of various species of flowers.

The `iris.db` file contains an SQL database with a dataset called `iris`. We will use it to compare familiar concepts between SQL and Pandas

## Importing Packages

In [ ]:
import pandas as pd     # The package we want to get to know, commonly imported as pd
import duckdb           # For using SQL on our database

## Reading data

### SQL

In [ ]:
# Establish connection to the database
con = duckdb.connect("iris.db")

sql_query = "select * from iris"

def show_query_results(query):
    display(con.execute(query).fetchdf())

show_query_results(sql_query)

In [ ]:
sql_query_large_sepal_length = """
select
    sepal_length
    , species
from
    iris
where
    sepal_length > 5
"""

show_query_results(sql_query_large_sepal_length)

### Pandas

In [ ]:
df = pd.read_csv("iris.csv")  # Pandas tables are called DataFrames or "df" for short

In [ ]:
df # Jupyter automatically displays the output of the last line in a cell

**Summary**
* Pandas offers out-of-the-box solutions for reading tabular data of various formats, e.g., `.csv`, `.xlsx`, `.pkl`, ...
* The function for for reading `.csv` files is called `_______`

## Columns vs. Rows

* An **SQL** query can be understood as a loop through each row in the dataset
    * For example, the query

        ```sql
        select
            sepal_length
            , species
        from
            iris
        where
            sepal_length > 5
        ```
        
        can be interpreted as going through each row and keeping only the ones where `sepal_length` is greater than `5`. An SQL table can be viewed a collection of rows.
* In **Pandas**, the paradigm shifts from **row**-based to **column**-based, i.e., a Pandas DataFrame is viewed as a collection of columns.

### Column Selection

Pandas uses two main datatypes:
* **pd.Series** - One-dimensional array of values
* **pd.DataFrame** - Collection of Series to form a table

In [ ]:
sepal_length_series = df["sepal_length"]  # Access a single column as a Series using []

type(sepal_length_series)

In [ ]:
sepal_length_series

In [ ]:
species_series = df["species"]

# Combine the two Series into a new DataFrame using pd.concat
df_new = pd.concat([sepal_length_series, species_series], axis=1)

df_new

We can also select multiple columns directly using a `list`

In [ ]:
selected_columns = ["sepal_length", "species"]

df_selected = df[selected_columns]

df_selected

**Summary**
* Pandas tables are called `________` and they are a collection of columns, so-called `_____`
* To access these columns, use the syntax `___` with the brackets containing either
    1. the name of a column (which returns a `_____`) or
    2. a list of multiple names (which returns a `________`)

### Row Selection

We now want to select rows from a DataFrame which fulfill a certain condition (`where` clause in SQL).

To accomplish this, Pandas uses Series of boolean values.

In [ ]:
df["sepal_length"] > 5  # This creates a Series of True/False values often called a "mask"

Now, to select only the `True` rows, Pandas allows us to use the same `[]` syntax as for selecting columns.

In [ ]:
mask_large_sepal_length = df["sepal_length"] > 5

df_large_sepal_length = df[mask_large_sepal_length]

df_large_sepal_length

Adding column selection:

In [ ]:
df[mask_large_sepal_length][selected_columns]

**Summary**
* To select rows of a DataFrame which fulfill a specific condition, we use Series with `_______` type, so-called `____`
* Once we have created such a Series, we can use it to select the rows of a DataFrame `df` via the syntax `____`

## Exercise

We have the following SQL query:
```sql
select
    sepal_width
    , petal_width
    , species
from
    iris
where
    sepal_length < 4.5
    and species = 'setosa'
```

**Exercise:**
1. Using what we learned above, we cannot yet fully refactor this SQL query to Pandas. What syntax are we missing?
2. Once we have gathered the relevant information, refactor the query.

In [ ]:
# Solution

## Outlook

We have now seen how we can translate basic SQL into Python, but **why** should we want to do that?

Let us look at two examples.

### 1. Quick Table Statistics

To get an initial view of a table, we can use `.info()` and `.describe()`

In [ ]:
df.info()

In [ ]:
df.describe()

The output of `.describe()` is again a Pandas DataFrame, so it can processed with the same tools!

### 2. Easy Column Creation

The same way we access columns, we can also create new ones.

In [ ]:
df["sepal_area"] = df["sepal_length"] * df["sepal_width"]

df["diff_sepal_petal_length"] = df["sepal_length"] - df["petal_length"]

df